## GCS Intervention — triggered guided plan

A single drone flies a straight AUTO mission heading north. When it reaches
waypoint `seq 3` the GCS **intervenes**: it switches the drone to GUIDED and runs
an `InterventionPlan` that guides it east to the target, then holds.

The intervention is a typed `Intervention(trigger, plan, firmware)`. The `MissionTrigger` fires on
a mission-sequence point, a time (`MissionTrigger(after=30)`), or both
(`mode="all"`/`"any"`); the plan is a multi-step guided sequence run GCS-side
against the vehicle's command channel. If the drone diverts east, the
GCS→Logic→SITL command path works.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import Intervention, MissionTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)

## Waypoints

In [ ]:
cruise_alt = 10.0
home_wp = ENU(x=0, y=0, z=0)
climb_wp = ENU(x=0, y=0, z=cruise_alt)
north_100 = ENU(x=0, y=20, z=cruise_alt)
north_200 = ENU(x=0, y=40, z=cruise_alt)
mission_wps = [home_wp, climb_wp, north_100, north_200]

## Vehicle

In [ ]:
sysid = 1
model = Model.IRIS


mission_path = DATA_PATH / "missions" / "gcs_intervention.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_path),
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)


## GCS

In [ ]:
gcs = SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}")
gcs.add_vehicle(vehicle)

### Intervnetion

When the drone reaches waypoint seq 3 (heading north_100 -> north_200) the GCS
takes over: it switches to GUIDED and runs a guided plan to the east target,
then holds. `gcs.intervene` requires the GCS to already monitor the vehicle.
The trigger can be a sequence point, a time (MissionTrigger(after=30)), N seconds
after the seq (MissionTrigger(seq=3, dwell=30)), or a combination (mode="all").
These conditions are monotone, so the GCS keeps control once it takes it; use a
`ProximityTrigger` for a guard that hands control back.

In [ ]:
target = ENU(x=20, y=20, z=cruise_alt)
gcs.intervene(
    vehicle,
    Intervention(
        trigger=MissionTrigger(seq=4),
        plan=InterventionPlan.from_relative_path(
            relative_path=[target],
            enu_origin=enu_origin,
            relative_home=home,
            firmware=model.firmware,
            land=False,
        ),
    ),
)

## Oracle

In [ ]:
orac = Oracle()
orac.add_gcs(gcs)


## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
target_marker = GazMarker(
    name="target",
    group="targets",
    pos=target,
    color=Color.GREEN,
)
gaz.markers.append(target_marker)
gaz.markers.append(origin_marker)


## Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=1,
    terminals=[SimProcess.LOGIC, SimProcess.GCS],
    speedup=3,
)

simulator.preview()


In [ ]:
simulator.run(timeout=75)


In [ ]:
orac.plot_trajectories(azim=-70, elev=15);
